# Imports/Settings

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
# 1. Стандартная библиотека
import os
import sys
import warnings
from pathlib import Path

# --- ДИНАМИЧЕСКИЙ РАСЧЕТ АБСОЛЮТНЫХ ПУТЕЙ ---
notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "notebooks":
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
import sweetviz as sv
from omegaconf import OmegaConf

# 3. Локальные модули
from src.core.data import get_data_source
from src.core.stats import GlobalStatCompiler
from src.core.features import TabularPreprocessor, FeatureEngineer
from src.core.utils import load_hydra_config
from src.eda.utils_eda import collapse_rare_categories

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
load_hydra_config.cache_clear()
# --- Инициализация Гидры ---
cfg = load_hydra_config()

print(f"Проект: {cfg.project_name} | Режим: Feature Engineering")
print(f"Проверка sample_pct: {cfg.data.sample_pct}")

run_name = Path(cfg.run_name)
reports_dir = Path(cfg.paths.reports_dir / run_name)
reports_dir.mkdir(parents=True, exist_ok=True)

# --- НАСТРОЙКИ ВИЗУАЛИЗАЦИИ ---
try:
    p_cfg = cfg.logging.plots
    plt.style.use(p_cfg.style)
    plt.rcParams.update({
        'figure.figsize': p_cfg.fig_size,
        'figure.dpi': p_cfg.dpi,
        'font.size': p_cfg.font_size,
        'axes.grid': p_cfg.grid,
        'axes.spines.top': p_cfg.spines_top,
        'axes.spines.right': p_cfg.spines_right
    })
    PLOT_ALPHA = p_cfg.alpha
except AttributeError:
    PLOT_ALPHA = 0.5
    print("Используются дефолтные стили Matplotlib.")

Проект: credit-risk-model | Режим: Feature Engineering
Проверка sample_pct: 1.0


# Data Loading

In [3]:
stats_compiler = GlobalStatCompiler(cfg, PROJECT_ROOT)
stats_compiler.load()
sql_injections = stats_compiler.get_sql_format_kwargs()

loader = get_data_source(cfg, PROJECT_ROOT)
df = loader.load(sql_injections)
target = cfg.data.tabular.target_col

print(f"Размер датасета: {df.shape}")
df.head(3)

ВНИМАНИЕ! Python прямо сейчас читает вот этот файл: C:\credit-risk-model\sql\aggregate\aggregate_credit_history_v_2_0.sql


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Размер датасета: (250000, 83)


,id,num_credits,max_rn,min_rn,rn_span,is_single_credit,rn_density,pclose_flag_share,pclose_flag_any,fclose_flag_share,...,pre_loans3060_nunique,pre_loans3060_last,pre_loans3060_changed_ever,pre_loans90_nunique,pre_loans90_share_global_mode,enc_loans_credit_type_nunique,enc_loans_credit_type_first,enc_loans_credit_type_last,enc_loans_credit_type_change_count,flag
0,100472,16,16,1,15,0,1.0,0.125,1,0.000000,...,1,5,0,1,1.0,2,4,4,8.0,0
1,100553,6,6,1,5,0,1.0,0.000,0,0.166667,...,1,5,0,1,1.0,3,3,3,4.0,0
2,100587,3,3,1,2,0,1.0,0.000,0,0.000000,...,1,5,0,1,1.0,3,2,0,2.0,0


In [4]:
target_col = cfg.data.tabular.target_col # 'event_value'
y_train = df[target_col].values
X_train_raw = df.drop(columns=[target_col])

# Feature Generation

In [5]:
tabular_cfg = cfg.get('data', {}).get('tabular', {})


In [8]:
def _safe_concat(df: pd.DataFrame, col1: str, col2: str, out_name: str, sep: str = "_"):
        """Безопасная строковая склейка."""
        if col1 in df.columns and col2 in df.columns:
            c1_str = df[col1].fillna(-999).astype(int).astype(str)
            c2_str = df[col2].fillna(-999).astype(int).astype(str)
            df[out_name] = c1_str + sep + c2_str

In [9]:
def transform(X: pd.DataFrame) -> pd.DataFrame:       
        X_transformed = X.copy()

        # =====================================================================
        # 1. МАКРО-ИНДЕКСЫ ТИПИЧНОСТИ И АНОМАЛИЙ
        # =====================================================================
        global_mode_cols = [c for c in X_transformed.columns if c.endswith('_share_global_mode')]
        if global_mode_cols:
            X_transformed['overall_typicalness_score'] = X_transformed[global_mode_cols].mean(axis=1)

        rare_bins_cols = [c for c in X_transformed.columns if c.endswith('_share_rare_bins')]
        if rare_bins_cols:
            X_transformed['overall_anomaly_score'] = X_transformed[rare_bins_cols].mean(axis=1)

        # =====================================================================
        # 2. ЖЕСТКИЕ ФЛАГИ ЭКСТРЕМУМОВ
        # =====================================================================
        if 'paym_row_share_code_0_mean' in X_transformed.columns:
            X_transformed['is_absolutely_perfect_payer'] = (
                X_transformed['paym_row_share_code_0_mean'] >= 0.95
            ).astype(np.int8)

        # НОВОЕ: Хронический должник (быстрая отсечка для класса 1)
        if 'share_any_overdue' in X_transformed.columns and 'share_serious_overdue' in X_transformed.columns:
            X_transformed['is_chronic_defaulter'] = (
                (X_transformed['share_any_overdue'] > 0.3) | 
                (X_transformed['share_serious_overdue'] > 0.0)
            ).astype(np.int8)

        # =====================================================================
        # 3. КАТЕГОРИАЛЬНЫЕ ЭВОЛЮЦИИ И СКЛЕЙКИ
        # =====================================================================
        _safe_concat(X_transformed, 'pre_util_last', 'pre_since_opened_last', 'cat_last_util_and_time')
        _safe_concat(X_transformed, 'enc_loans_credit_type_last', 'pre_loans_credit_limit_last', 'cat_last_type_and_limit')
        _safe_concat(X_transformed, 'pre_loans_credit_limit_first', 'pre_loans_credit_limit_last', 'cat_limit_transition')
        _safe_concat(X_transformed, 'enc_loans_credit_type_first', 'enc_loans_credit_type_last', 'cat_type_transition')
        
        # НОВОЕ: Эволюция статуса кредита
        _safe_concat(X_transformed, 'first_enc_loans_credit_status', 'last_enc_loans_credit_status', 'cat_status_transition')

        # НОВОЕ: Прошлое (Доля просрочек) + Настоящее (Утилизация)
        if 'share_any_overdue' in X_transformed.columns and 'pre_util_last' in X_transformed.columns:
            # Превращаем непрерывную долю просрочек в понятные текстовые бакеты
            history_bins = pd.cut(
                X_transformed['share_any_overdue'], 
                bins=[-np.inf, 0.0, 0.2, np.inf], 
                labels=["Clean", "Mild", "Frequent"]
            ).astype(str)
            
            util_str = X_transformed['pre_util_last'].fillna(-999).astype(int).astype(str)
            X_transformed['cat_util_and_history'] = util_str + "_" + history_bins

        # НОВОЕ: Матрица кредитного опыта (Разнообразие типов + Общее число)
        if 'distinct_enc_loans_credit_type_count' in X_transformed.columns and 'num_credits' in X_transformed.columns:
            types_str = X_transformed['distinct_enc_loans_credit_type_count'].fillna(0).astype(int).astype(str)
            loans_str = X_transformed['num_credits'].fillna(0).astype(int).astype(str)
            X_transformed['cat_diversity_profile'] = types_str + "types_" + loans_str + "loans"

        return X_transformed

In [10]:
X_train = transform(X_train_raw)

In [11]:
preprocessor = TabularPreprocessor(cfg)
X_train_processed = preprocessor.fit_transform(X_train)

In [16]:
X_train_encoded = X_train_processed.copy()
cat_cols = X_train_encoded.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    # Преобразуем в строку перед кодированием на всякий случай
    X_train_encoded[cat_cols] = encoder.fit_transform(X_train_encoded[cat_cols].astype(str))

In [17]:
mi_scores = mutual_info_classif(X_train_encoded, y_train)

mi_df = pd.DataFrame({'Feature': X_train_encoded.columns, 'MI': mi_scores})
print(mi_df.sort_values(by='MI', ascending=False))

                                         Feature        MI
70                         pre_loans3060_nunique  0.061219
83                          cat_limit_transition  0.054398
10                          loans3060_zero_share  0.051371
5                                pclose_flag_any  0.050125
7                                fclose_flag_any  0.050059
..                                           ...       ...
44            pre_since_opened_share_global_mode  0.001531
67  pre_loans_credit_cost_rate_share_global_mode  0.001381
55        pre_loans_credit_limit_share_rare_bins  0.001309
71                    pre_loans3060_changed_ever  0.001206
3                               is_single_credit  0.000986

[86 rows x 2 columns]


In [18]:
eda_df = X_train_encoded.copy()
eda_df[target_col] = y_train

# Схлопываем редкие категории, чтобы Sweetviz построил красивые графики без каши
cat_cols = eda_df.select_dtypes(exclude=[np.number]).columns.tolist()
eda_df_collapsed = collapse_rare_categories(eda_df, cat_cols, top_n=12)

feature_config = sv.FeatureConfig()
feature_config.force_num = [target_col]
feature_config.force_cat = []

report = sv.analyze(eda_df_collapsed, target_feat=target_col, feat_cfg=feature_config)
output_path = reports_dir / "eda_featuredv2.0.html"
report.show_html(filepath=str(output_path), open_browser=True)

                                             |          | [  0%]   00:00 -> (? left)

Report reports\p_v0.0.1__f_v0.0.1__m_ncatboost_v0.0.1\eda_featuredv2.0.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


# Выводы по Feature Engineering

## Какие категории фич были добавлены
- Парсинг площади экрана из разрешения
- Глобальная региональность (Россия, страны СНГ, остальной мир)
- календарные и временные фичи (день,месяц, день недели, время суток)
- математика вокруг кликов (время между кликами, количество кликов)
- комбинирование категории девайса и бренда
- внешние фичи связанные с демографией и статистикой городов

## Идеи для новых фичей
- таргет энкодер по различным фичам таким как(категория девайса, код машины(марка+модель), geo_zone, geo_city) Однако я буду использовать catboost т.к. очень много категориальных фич, поэтому это действие не нужно
- аггрегирование математических статистик по фичам таким как (модель телефона, операционная система, модель автомобиля) таких фичей как (среднее количество кликов, screen_area, потенциально любых числовых фичей)
- попробовать соединить город и код машины т.к. вероятно есть связь между классом машины и городом (допустим внедорожник или машина представительского класса)
- поведенческие сдвиги (такие как total_hits_count пользователя / (медианное количество кликов для его device_brand))
- динамика внутри сессии (например is_fast_session если среднее между кликами меньше 500мс то это бот и у него конверсии не будет)

## Влияние фич предварительное (по mi_score)

| Feature | MI |
| :--- | :--- |
| unique_event_actions | 0.027409 |
| geo_zone | 0.023055 |
| total_car_views | 0.022329 |
| user_vs_city_car_interest | 0.022149 |
| total_hits_count | 0.021664 |
| total_events_count | 0.021111 |
| geo_country | 0.018660 |
| device_category | 0.018327 |
| is_mobile_device | 0.017867 |
| has_metro | 0.017524 |
| car_view_ratio | 0.016754 |
| unique_cars_viewed | 0.013276 |
| utm_keyword | 0.011159 |


Новая фича user_vs_city_car_interest, созданная на стыке логов и демографии городов, показала высокий MI_Score (0.022149), заняв 4-е место в общем рейтинге. Это доказывает успешность гипотезы об обогащении данных внешними макро-показателями рынков.

## Предварительно бесполезные фичи (по mi_score)

| Feature | MI |
| :--- | :--- |
| first_hit_time_ms | 0.000000 |
| is_first_hit_car_view | 0.000000 |